# TSIS &mdash; Lab 4

## Optimizing AI agents for production

**Demo:** Multi-Vendor Supply Chain & Procurement Risk Analyst AI Agent

In this hands-on lab, you will diagnose and fix four real-world production bottlenecks in a (mock) enterprise LangChain procurement agent:

1. **Sequential I/O Bottleneck** → Async tools + `asyncio.gather` multi-vendor fan-out (`Exercise 1`)
2. **Redundant Slow Lookups vs. Volatile Data** → Selective TTL caching via `AgentMiddleware` (`Exercise 2`)
3. **Cascading Downstream Outages (503 / 429)** → Circuit breaker + retry composition (`Exercise 3`)
4. **RAG Context / vLLM KV-Cache Bloat** → Payload compaction middleware before state insertion (`Exercise 4`)

**Scenario (`PRIMARY_PO` / `PO-2026-0412`):** *Carpathia Precision Motors S.A.* needs 60,000 automotive traction-inverter MCUs delivered to Constanța (`ROCND`) and must choose whether to keep incumbent `VND-101` (Shanghai) or switch to `VND-204` (Hamburg) or `VND-309` (Izmir).


To begin with, edit the next cell and fill in your API keys (for the LLM endpoint and for LangSmith).

In [ ]:
API_KEY = "FILL IN YOUR API KEY HERE"
BASE_URL = "https://api.llm.acc.unibuc.ro/v1"
MODEL_NAME = "Qwen3.8-27B"

LANGSMITH_API_KEY = "FILL IN YOUR LANGSMITH API KEY HERE"
LANGSMITH_API_URL = "https://eu.api.smith.langchain.com/"

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = API_KEY
os.environ["OPENAI_API_BASE"] = BASE_URL
os.environ["OPENAI_MODEL"] = MODEL_NAME

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGSMITH_AGENT_ID"] = "curs-tsis"
os.environ["LANGSMITH_AGENT_ENVIRONMENT"] = "local"
os.environ["LANGSMITH_API_URL"] = LANGSMITH_API_URL

## Environment Setup & In-Notebook Mock Server

Before running the cells below:
1. Copy `.env.example` to `.env` and fill in `OPENAI_BASE_URL`, `OPENAI_API_KEY`, and `OPENAI_MODEL` for your LiteLLM / vLLM gateway.
2. Make sure Jupyter is running inside the `uv` environment (`uv run jupyter lab notebooks/supply_chain_lab.ipynb`).


In [ ]:
from supply_chain_demo.config import build_chat_model, load_settings
from supply_chain_demo.data.scenarios import FOLLOWUP_PO, PRIMARY_PO
from supply_chain_demo.benchmark import (
    compare_reports,
    print_report,
    render_waterfall,
)

settings = load_settings()
print(f"LiteLLM/vLLM Base URL : {settings.openai_base_url}")
print(f"Model                 : {settings.openai_model}")
print(f"Mock API Target       : {settings.mock_api_base_url}")

# Official OpenAI-compatible client configured with use_responses_api=False
model = build_chat_model(settings)
reports = []


In [ ]:
# Start the local FastAPI mock enterprise microservices inside this Jupyter session.
# `start_mock_server` is idempotent: re-running this cell reuses the running background server
# and resets recorded metrics and injected faults cleanly.
from supply_chain_demo.server.launcher import start_mock_server

server = start_mock_server(host="127.0.0.1", port=8000)
print(f"Mock Enterprise Server healthy at {server.url}")
print("Initial server faults:", server.reset_faults())


## Section 1: Probing the 4 Enterprise Microservices

Let's inspect the latency, `Cache-Control` headers, and payload size of each backend service directly before running our agent:
- **ERP SQLite (`scorecard`)**: ~600 ms, cacheable for 24h (`max-age=86400`)
- **Live Freight API (`rates`)**: 600–1,100 ms jitter, volatile spot quote (`no-store`)
- **Maritime Disruption RAG (`disruptions`)**: ~800 ms TF-IDF search returning **~2,000 tokens** of noisy logs & manifests
- **Compliance Policy (`policies`)**: ~400 ms, static & immutable (`max-age=31536000, immutable`)


In [ ]:
import time
import httpx

probes = [
    ("ERP Scorecard", f"{server.url}/api/erp/vendors/VND-204/scorecard", {}),
    ("Freight Spot Rate", f"{server.url}/api/freight/rates", {"origin_port": "DEHAM", "dest_port": "ROCND"}),
    ("Disruption News RAG", f"{server.url}/api/news/disruptions", {"region": "south_china_sea_red_sea", "top_k": 3}),
    ("Compliance Policy", f"{server.url}/api/compliance/policies/automotive_semiconductors", {}),
]

for label, url, params in probes:
    t0 = time.perf_counter()
    resp = httpx.get(url, params=params, timeout=10.0)
    ms = (time.perf_counter() - t0) * 1000.0
    cache_hdr = resp.headers.get("cache-control", "<none>")
    print(f"{label:22s} | status={resp.status_code} | {ms:6.1f} ms | chars={len(resp.text):5d} | Cache-Control={cache_hdr}")

news_json = httpx.get(
    f"{server.url}/api/news/disruptions",
    params={"region": "south_china_sea_red_sea", "top_k": 3},
    timeout=10.0,
).json()
print(f"\nNews payload approx_tokens={news_json['approx_tokens']} (first 300 chars of bulletin #1):")
print(news_json["bulletins"][0]["raw_chunk_text"][:300])


## Section 2: Naive Baseline (`create_agent` + Blocking Sync Tools)

In `src/supply_chain_demo/agent/baseline.py`, `evaluate_purchase_order_sync` loops sequentially over the 3 candidate vendors, and `src/supply_chain_demo/tools/sync_tools.py` uses a single serialized `httpx.Client` connection.

Run the baseline now and examine both the **Run Report** and the **ASCII HTTP Waterfall** (`3 vendors × 4 tools = 12 sequential HTTP calls`, ~8–10 seconds before synthesis!).


In [ ]:
from supply_chain_demo.agent.baseline import evaluate_purchase_order_sync

baseline_decision, baseline_report = evaluate_purchase_order_sync(
    PRIMARY_PO,
    model,
    server.url,
)
reports.append(baseline_report)

print("=== BASELINE PROCUREMENT DECISION ===")
print(baseline_decision)
print()
print_report(baseline_report)
print()
print(render_waterfall(baseline_report))


## Section 3 (Exercise 1): Async Parallelization (`httpx.AsyncClient` + `asyncio.gather`)

Look at the waterfall above: every HTTP call waited for the previous one to finish!

**Your tasks:**
1. Open `src/supply_chain_demo/tools/async_tools.py` and implement all four `async def` tools using `await client.get(...)`.
2. Open `src/supply_chain_demo/agent/optimized.py` and implement:
   - `evaluate_vendor_async` using `await agent.ainvoke(...)`
   - `evaluate_purchase_order_async` using `asyncio.gather(...)` across `po.candidate_vendors` and `await model.ainvoke(...)` for final synthesis.

> **Jupyter Async Tip:** Jupyter notebooks already run an active `asyncio` event loop. Always use **top-level `await`** in notebook cells (e.g., `await evaluate_purchase_order_async(...)`) — **never** call `asyncio.run(...)` inside Jupyter!


In [ ]:
import httpx
from supply_chain_demo.tools.async_tools import create_async_tools
from supply_chain_demo.agent.optimized import evaluate_purchase_order_async

try:
    async with httpx.AsyncClient(
        limits=httpx.Limits(max_connections=20, max_keepalive_connections=10),
        timeout=30.0,
    ) as async_client:
        async_tools = create_async_tools(async_client, server.url)
        ex1_decision, ex1_report = await evaluate_purchase_order_async(
            PRIMARY_PO,
            model,
            async_tools,
            server.url,
            label="Ex 1: Async Parallel",
        )

    reports.append(ex1_report)
    print_report(ex1_report)
    print()
    print(render_waterfall(ex1_report))
except NotImplementedError as exc:
    print(f"[Exercise 1 Pending] {exc}")


## Section 4 (Exercise 2): Selective TTL Caching (`SelectiveCacheMiddleware`)

Even after parallelizing, evaluating two back-to-back purchase orders (`PRIMARY_PO` and `FOLLOWUP_PO`) re-fetches **static** compliance rules and **daily-cached** ERP vendor scorecards for overlapping suppliers (`VND-204`, `VND-309`)!

**Your task:**
Implement `SelectiveCacheMiddleware.awrap_tool_call` in `src/supply_chain_demo/middleware/exercises.py`:
- Cache `get_compliance_policy` infinitely (`ttl=None`)
- Cache `get_vendor_historical_scorecard` for 24h (`ttl=86400.0`)
- **Bypass** caching for `check_live_freight_rates` (`ttl=0.0`)
- On a cache hit, return a fresh `ToolMessage(content=..., tool_call_id=request.tool_call["id"], name=tool_name)` bound to the **current** `tool_call_id`!

Run both `PRIMARY_PO` (cold cache) and `FOLLOWUP_PO` (warm cache) below to verify that Compliance & overlapping ERP calls hit the cache while live Freight rates still reach the server.


In [ ]:
from supply_chain_demo.middleware.exercises import SelectiveCacheMiddleware

cache_mw = SelectiveCacheMiddleware()

try:
    async with httpx.AsyncClient(
        limits=httpx.Limits(max_connections=20, max_keepalive_connections=10),
        timeout=30.0,
    ) as async_client:
        async_tools = create_async_tools(async_client, server.url)

        _, cold_report = await evaluate_purchase_order_async(
            PRIMARY_PO,
            model,
            async_tools,
            server.url,
            middleware=[cache_mw],
            label="Ex 2a: Cache Cold (PO-0412)",
        )
        _, warm_report = await evaluate_purchase_order_async(
            FOLLOWUP_PO,
            model,
            async_tools,
            server.url,
            middleware=[cache_mw],
            label="Ex 2b: Cache Warm (PO-0419)",
        )

    reports.append(warm_report)
    print(
        f"Cache stats -> hits={cache_mw.hits}, misses={cache_mw.misses}, bypassed={cache_mw.bypassed}"
    )
    print_report(warm_report)
    print()
    print(render_waterfall(warm_report))
except NotImplementedError as exc:
    print(f"[Exercise 2 Pending] {exc}")


## Section 5 (Exercise 3): Circuit Breaker & Retries (`FreightCircuitBreakerMiddleware`)

External shipping APIs suffer outages (`503 Service Unavailable`) and rate limits (`429 Too Many Requests`). Retrying blindly against a dead upstream wastes seconds per vendor (and can crash unguarded tool loops).

**Your task:**
Implement `FreightCircuitBreakerMiddleware.awrap_tool_call` in `src/supply_chain_demo/middleware/exercises.py` with `CLOSED → OPEN → HALF_OPEN` transitions, and compose it with LangChain's built-in `ToolRetryMiddleware`:

```python
middleware=[
    cache_mw,
    cb_mw,
    ToolRetryMiddleware(max_retries=1, tools=["check_live_freight_rates"], initial_delay=0.3, on_failure="continue"),
]
```

> **Middleware ordering question:** Why must `FreightCircuitBreakerMiddleware` be placed **before** `ToolRetryMiddleware`? What happens if you reverse them?

Let's inject a live `503` outage (`1,200 ms` simulated timeout) via `server.set_faults(...)`, verify fail-fast degradation, and then restore healthy server defaults!


In [ ]:
from langchain.agents.middleware import ToolRetryMiddleware
from supply_chain_demo.middleware.exercises import FreightCircuitBreakerMiddleware

# Inject a simulated upstream 503 outage on the Freight service
faults_state = server.set_faults(freight_failure_mode="outage_503", freight_failure_latency_ms=1200)
print("Injected outage:", faults_state)

cb_mw = FreightCircuitBreakerMiddleware(
    target_tools=("check_live_freight_rates",),
    failure_threshold=2,
    recovery_timeout_s=30.0,
)
retry_mw = ToolRetryMiddleware(
    max_retries=1,
    tools=["check_live_freight_rates"],
    initial_delay=0.25,
    on_failure="continue",
)

try:
    async with httpx.AsyncClient(
        limits=httpx.Limits(max_connections=20, max_keepalive_connections=10),
        timeout=30.0,
    ) as async_client:
        async_tools = create_async_tools(async_client, server.url)
        outage_decision, outage_report = await evaluate_purchase_order_async(
            PRIMARY_PO,
            model,
            async_tools,
            server.url,
            middleware=[cache_mw, cb_mw, retry_mw],
            label="Ex 3: Outage + Circuit Breaker",
        )

    reports.append(outage_report)
    print(f"Circuit state={cb_mw.state.value}, short_circuited_calls={cb_mw.short_circuited_calls}")
    print_report(outage_report)
    print()
    print(render_waterfall(outage_report))
    print("\n=== DEGRADED-MODE DECISION ===")
    print(outage_decision[:800])
except NotImplementedError as exc:
    print(f"[Exercise 3 Pending] {exc}")
finally:
    server.reset_faults()
    print("\nRestored server faults:", server.reset_faults())


## Section 6 (Exercise 4): KV-Cache & Context Bloat Compaction (`NewsContextCompactionMiddleware`)

Each `scan_port_disruption_news` call returns ~2,000 tokens of raw AIS telemetry logs, vessel anchorage manifests, and legal boilerplate — inflating the prompt prefix and vLLM KV-cache footprint by ~6,000 tokens per PO run!

**Your task:**
Implement `NewsContextCompactionMiddleware.awrap_tool_call` in `src/supply_chain_demo/middleware/exercises.py` to parse the JSON response from `scan_port_disruption_news` and strip `raw_chunk_text`, keeping only the structured decision fields (`bulletin_id`, `region`, `published_at`, `alert_severity`, `estimated_delay_days`, `summary`).

After running the cell below, inspect `compaction_mw.chars_saved` alongside the prompt-token reduction visible in your configured observability backend (**LangSmith**, **LangFuse**, or **OpenTelemetry**), and compare this **pre-state tool output compaction** against LangChain's built-in `SummarizationMiddleware` (which compresses older turns *after* history has already grown).


In [ ]:
from supply_chain_demo.middleware.exercises import NewsContextCompactionMiddleware

compaction_mw = NewsContextCompactionMiddleware(target_tool="scan_port_disruption_news")
fresh_cache_mw = SelectiveCacheMiddleware()
fresh_cb_mw = FreightCircuitBreakerMiddleware()

try:
    async with httpx.AsyncClient(
        limits=httpx.Limits(max_connections=20, max_keepalive_connections=10),
        timeout=30.0,
    ) as async_client:
        async_tools = create_async_tools(async_client, server.url)
        final_decision, compacted_report = await evaluate_purchase_order_async(
            PRIMARY_PO,
            model,
            async_tools,
            server.url,
            middleware=[fresh_cache_mw, fresh_cb_mw, compaction_mw],
            label="Ex 4: Full Optimized + Compacted",
        )

    reports.append(compacted_report)
    print(f"News payload chars saved by compaction middleware: {compaction_mw.chars_saved:,}")
    print_report(compacted_report)
    print()
    print(render_waterfall(compacted_report))
except NotImplementedError as exc:
    print(f"[Exercise 4 Pending] {exc}")


## Section 7: End-to-End Benchmark Comparison & Server Shutdown

Let's compare all recorded runs side-by-side and cleanly shut down the background FastAPI server.


In [ ]:
compare_reports(reports)
server.stop()
print("MockEnterpriseServer stopped.")
